# Original 2023 notebook (reconstructed)

This is the code from my 2023 BSc final-year project at Universiti Sains Malaysia,
*Machine Learning Applications for the James Clerk Maxwell Telescope* (June 2023).

The original Google Colab notebook no longer exists. This copy was transcribed from the code
screenshots in Appendix A of the thesis. The code is kept exactly as it appeared there,
including the duplicated `base_dir` line. Any cells before cell [2] (for example, mounting
Google Drive or unzipping the data) were not shown in the thesis and are not included.

The dataset was 150 PNG images from the JCMT Transient Survey (85 star-forming, 65
non-star-forming), stored in `/content/Data/train` and `/content/Data/validation`, with one
subfolder per class. The images are not available, so this notebook documents the code
but cannot be rerun as it is.

It is kept for reference only. The rebuilt 2026 project is in the `sfcnn/` package; see the
main README for what changed and why.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

## Set up the directories

In [ ]:
base_dir = '/content/Data'
base_dir = '/content/Data'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

## Data augmentation and normalization

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

# Specify the image dimensions and batch size
img_width, img_height = 150, 150
batch_size = 32

# Generate the training and validation data from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary'
)

## Build the CNN model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])

## Compile the model

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Train the model

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

## Evaluate the model on the validation set

In [ ]:
loss, accuracy = model.evaluate(validation_generator)
print('Validation loss:', loss)
print('Validation accuracy:', accuracy)

## Plot the training and validation accuracy

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])
plt.show()

## Looking back (2026)

What I would now do differently, and what the rebuilt project does instead:

- **PNG conversion** discarded the physical units and the noise map. The rebuild reads the
  FITS maps directly and uses S/N and normalised-flux channels.
- **Labels** were not clearly defined. The rebuild labels each core by cross-matching it
  against a YSO catalogue (Dunham et al. 2015).
- **Evaluation:** the reported metrics came from the validation set, which was also used
  to judge training, and from a random split of only 150 images. The rebuild uses a separate
  test set and leave-one-cloud-out cross-validation.
- **No baselines.** The rebuild compares the CNN against peak S/N, logistic regression and a
  random forest.
- **Accuracy only.** With unequal classes, ROC-AUC and PR-AUC are more informative.